In [1]:
import random
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

In [2]:
CITIES = np.array([
    [6734, 1453], [2233, 10], [5530, 1424], [401, 841], [3082, 1644],
    [7608, 4458], [7573, 3716], [7265, 1268], [6898, 1885], [1112, 2049],
    [5468, 2606], [5989, 2873], [4706, 2674], [4612, 2035], [6347, 2683],
    [6107, 669], [7611, 5184], [7462, 3590], [7732, 4723], [5900, 3561],
    [4483, 3369], [6101, 1110], [5199, 2182], [1633, 2809], [4307, 2322],
    [675, 1006], [7555, 4819], [7541, 3981], [3177, 756], [7352, 4506],
    [7545, 2801], [3245, 3305], [6426, 3173], [4608, 1198], [23, 2216],
    [7248, 3779], [7762, 4595], [7392, 2244], [3484, 2829], [6271, 2135],
    [4985, 140], [1916, 1569], [7280, 4899], [7509, 3239], [10, 2676],
    [6807, 2993], [5185, 3258], [3023, 1942]
])

OPTIMAL_SOLUTION = 35000.00

In [3]:
def set_seed(seed=42):
    random.seed(seed)           
    np.random.seed(seed)

In [4]:
def create_route(num_cities):
    return np.random.permutation(num_cities)

def calculate_distance(cities):
    num_cities = len (cities)
    distance_matrix = np.full((num_cities, num_cities), np.inf)
    for i in range(num_cities):
        for j in range(num_cities):
            distance_matrix[i, j] = np.linalg.norm(cities[i] - cities[j])
    return distance_matrix

def calculate_route_distance(route, distance_matrix):
    distance = 0
    for i in range(len(route) - 1):
        distance += distance_matrix[route[i], route[i + 1]]
    distance += distance_matrix[route[-1], route[0]]
    return distance

def selection_parents(next_generation):
    selected_parents = random.sample(next_generation, 2)
    return selected_parents[0], selected_parents[1]

In [5]:
def original_crossover(parent1, parent2):
    size = len(parent1)
    start, end = sorted(random.sample(range(size), 2))
    child = [None] * size
    child[start:end] = parent1[start:end]
    pointer = end

    for city in parent2:
        if city not in child:
            while child[pointer % size] is not None:
                pointer += 1
            child[pointer % size] = city
    return child

def original_mutate(child, mutation_rate):
    if random.random() < mutation_rate:
        i, j = random.sample(range(len(child)), 2)
        child[i], child[j] = child[j], child[i]
    return child

def build_edge_table(parent1, parent2):
    size = len(parent1)

    edge_table = {i: set() for i in range(size)}
    
    for p in [parent1, parent2]:
        for i in range(size):
            edge_table[p[i]].add(p[(i - 1) % size])
            edge_table[p[i]].add(p[(i + 1) % size])
    
    return edge_table

def proposed_crossover(parent1, parent2):
    size = len(parent1)
    edge_table = build_edge_table(parent1, parent2)
    child = []
    current_city = random.choice([parent1[0], parent2[0]])
    
    while len(child) < size:
        child.append(current_city)
        
        for neighbors in edge_table.values():
            neighbors.discard(current_city)    
            
        neighbors_of_current = edge_table[current_city]
        
        if neighbors_of_current:
            min_size = min(len(edge_table[n]) for n in neighbors_of_current)
            candidates = [n for n in neighbors_of_current if len(edge_table[n]) == min_size]
            next_city = random.choice(candidates)
        else:
            remaining_cities = [c for c in range(size) if c not in child]
            if remaining_cities:
                next_city = random.choice(remaining_cities)
            else:
                break
                
        current_city = next_city
        
    return child

def proposed_mutate(child, mutation_rate):
    if random.random() < mutation_rate:
        i, j = sorted(random.sample(range(len(child)), 2))
        child[i:j] = reversed(child[i:j])
    return child


In [41]:
class TSPVisualizer:
    def __init__(self, cities_coords):
        self.coords = cities_coords
        self.fig, self.ax = plt.subplots(figsize=(10, 7))
       
        self.line, = self.ax.plot([], [], 'o-', mfc='red', ms=4, lw=1, color='#3498db')
        self.text = self.ax.text(0.02, 0.95, '', transform=self.ax.transAxes, 
                                 fontsize=10, bbox=dict(facecolor='white', alpha=0.7))
        
        self.ax.set_title("Evolución de Ruta TSP", fontsize=16, fontweight='bold', pad=20)
        self.ax.set_xlabel("Coordenada X", fontsize=12)
        self.ax.set_ylabel("Coordenada Y", fontsize=12)
        self.ax.grid(True, linestyle='--', alpha=0.4)
        
        margin = 0.05
        x_min, y_min = self.coords.min(axis=0)
        x_max, y_max = self.coords.max(axis=0)
        dx, dy = x_max - x_min, y_max - y_min
        self.ax.set_xlim(x_min - dx*margin, x_max + dx*margin)
        self.ax.set_ylim(y_min - dy*margin, y_max + dy*margin)

    def update(self, best_route, generation, distance):
        route_indices = list(best_route) + [best_route[0]]
        ordered_coords = self.coords[route_indices]
        
        x = ordered_coords[:, 0]
        y = ordered_coords[:, 1]
        
        self.line.set_data(x, y)
        self.text.set_text(f"Generación: {generation}\nDistancia: {distance:,.2f}")
        
        self.fig.canvas.draw()
        self.fig.canvas.flush_events()
        return self.line, self.text

def save_tsp_animation(history_routes, history_distances, coords, filename='results/proposed_algorithm_tsp_evolution.mp4'):
    coords = np.array(coords)
    fig, ax = plt.subplots(figsize=(10, 7))
    line, = ax.plot([], [], 'o-', lw=1, ms=3, color='#3498db', mfc='red')
    
    ax.set_xlim(coords[:,0].min()-100, coords[:,0].max()+100)
    ax.set_ylim(coords[:,1].min()-100, coords[:,1].max()+100)

    ax.set_xlabel("Coordenada X")
    ax.set_ylabel("Coordenada Y")

    def animate(i):
        route = history_routes[i]
        route_idx = list(route) + [route[0]]
        ordered = coords[route_idx]
        line.set_data(ordered[:, 0], ordered[:, 1])
        ax.set_title(f"Generación: {i} | Distancia: {history_distances[i]:,.2f}", fontsize=14)
        return line,

    ani = FuncAnimation(fig, animate, frames=len(history_routes), interval=20, blit=True)
    
    try:
        ani.save(filename, writer='ffmpeg', fps=20)
    except Exception as e:
        print(f"Error al guardar: {e}.")
    
    plt.close()

In [37]:
def TSP_genetic_algorithm(cities, population_size=100, num_generations=2000, mutation_rate=0.2, stop_distance = OPTIMAL_SOLUTION, mode="original"):
    visualizer = TSPVisualizer(cities)
    set_seed()
    distance_matrix = calculate_distance(cities) 
    population = [create_route(len(cities)) for _ in range(population_size)]

    history_routes = []
    history_distances = []

    for generation in range(num_generations):
        population = sorted(population, key=lambda route: calculate_route_distance(route, distance_matrix))
        best_route_in_gen = population[0]
        best_distance_in_gen = calculate_route_distance(best_route_in_gen, distance_matrix)

        history_routes.append(best_route_in_gen.copy())
        history_distances.append(best_distance_in_gen)

        if generation % 10 == 0:
            visualizer.update(best_route_in_gen, generation, best_distance_in_gen)
            plt.pause(0.01)

        if calculate_route_distance(population[0], distance_matrix) > stop_distance:
            next_generation = population[: population_size // 2]

            for _ in range(population_size // 2):
                parent1, parent2 = selection_parents(next_generation)
                child = original_crossover(parent1, parent2) if mode == "original" else proposed_crossover(parent1, parent2)
                next_generation.append(original_mutate(child, mutation_rate) if mode == "original" else proposed_mutate(child, mutation_rate))

            population = next_generation
        
        else:
            save_tsp_animation(history_routes, history_distances, cities)
            return population[0], calculate_route_distance(population[0], distance_matrix), generation
    
    best_route = population[0]

    save_tsp_animation(history_routes, history_distances, cities)
    
    return best_route, calculate_route_distance(best_route, distance_matrix), generation

In [40]:
%matplotlib qt5
TSP_genetic_algorithm(CITIES, mutation_rate=0.2, mode="proposed")

([np.int32(16),
  np.int32(18),
  36,
  np.int32(5),
  np.int32(35),
  np.int32(45),
  32,
  19,
  np.int32(46),
  np.int32(20),
  np.int32(31),
  23,
  np.int32(44),
  np.int32(34),
  np.int32(3),
  25,
  np.int32(9),
  41,
  np.int32(1),
  28,
  4,
  np.int32(47),
  np.int32(38),
  np.int32(24),
  np.int32(13),
  np.int32(12),
  22,
  10,
  11,
  14,
  np.int32(39),
  2,
  np.int32(33),
  np.int32(40),
  15,
  21,
  np.int32(7),
  np.int32(0),
  np.int32(8),
  np.int32(37),
  np.int32(30),
  np.int32(43),
  np.int32(17),
  6,
  np.int32(27),
  np.int32(29),
  np.int32(42),
  np.int32(26)],
 np.float64(34648.62049933515),
 331)